# Stage 1: Binary Rain/No-Rain Classifier
**ML-Based Acoustic Rain Gauge**

- **Train**: Nov 2023 – Dec 2025
- **Test**: Jan 2026 – Jun 2026
- **Model**: XGBoost Binary Classifier
- **Features**: 112 acoustic features per 3-min window

In [ ]:
# Cell 1 — List input files
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Step 1: Imports

In [ ]:
# Cell 2 — Imports
import numpy as np
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_recall_curve,
    roc_curve
)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

OUTPUT_DIR = '/kaggle/working'
print("Imports done.")

## Step 2: Load & Combine Monthly Parquet Files

> **Important**: Update  to match your Kaggle dataset name. Check  in Cell 1 output.

In [ ]:
# Cell 3 — Load & combine
# !! UPDATE THIS to match your dataset slug from Cell 1 output
FEATURES_DATASET_PATH = '/kaggle/input/features/'

files = sorted(glob.glob(f"{FEATURES_DATASET_PATH}features_*.parquet"))
files = [f for f in files if 'combined' not in f]

print(f"Found {len(files)} monthly parquet files:")
for f in files:
    print(f"  {os.path.basename(f)}")

dfs = [pd.read_parquet(f) for f in files]
df = pd.concat(dfs, ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"
Total rows : {len(df)}")
print(f"Total cols : {df.shape[1]}")
print(f"Date range : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"
Class balance:")
print(df['rain'].value_counts())
print(df['rain'].value_counts(normalize=True).mul(100).round(2))

## Step 3: Define Features & Time-Based Train/Test Split

> **Critical**: No random split — temporal integrity required to prevent data leakage.

In [ ]:
# Cell 4 — Feature cols & time-based split
META_COLS = ["timestamp", "rainfall_mm", "wav_count",
             "rain", "n_clips_used", "0_mean", "0_std"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
TARGET = "rain"

print(f"Feature columns: {len(FEATURE_COLS)}")

# Time-based split
CUTOFF = pd.Timestamp("2026-01-01")
train = df[df["timestamp"] < CUTOFF].copy()
test  = df[df["timestamp"] >= CUTOFF].copy()

# Drop null rows
train = train.dropna(subset=FEATURE_COLS).reset_index(drop=True)
test  = test.dropna(subset=FEATURE_COLS).reset_index(drop=True)

print(f"
Train: {len(train)} rows  ({train['timestamp'].min().date()} → {train['timestamp'].max().date()})")
print(f"Test:  {len(test)} rows   ({test['timestamp'].min().date()} → {test['timestamp'].max().date()})")

print(f"
Train class balance:")
print(train[TARGET].value_counts())
print(train[TARGET].value_counts(normalize=True).mul(100).round(2))

print(f"
Test class balance:")
print(test[TARGET].value_counts())
print(test[TARGET].value_counts(normalize=True).mul(100).round(2))

## Step 4: Scale Features & Compute Class Imbalance Weight

In [ ]:
# Cell 5 — Scale features
X_train = train[FEATURE_COLS].values
y_train = train[TARGET].values
X_test  = test[FEATURE_COLS].values
y_test  = test[TARGET].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

joblib.dump(scaler, f"{OUTPUT_DIR}/scaler_stage1.pkl")
print("Scaler saved → scaler_stage1.pkl")

# Class imbalance weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"Neg (no rain): {neg}")
print(f"Pos (rain)   : {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

## Step 5: Train XGBoost Binary Classifier

In [ ]:
# Cell 6 — Train XGBoost
clf = xgb.XGBClassifier(
    n_estimators          = 1000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    gamma                 = 0.1,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = "auc",
    early_stopping_rounds = 50,
    use_label_encoder     = False,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 1,
)

clf.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=100,
)

print(f"
Best iteration: {clf.best_iteration}")

## Step 6: Threshold Tuning

> Default threshold (0.5) is suboptimal for imbalanced data. We find the F1-optimal threshold from the precision-recall curve.

In [ ]:
# Cell 7 — Threshold tuning
y_prob = clf.predict_proba(X_test_scaled)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print("Default threshold (0.5):")
y_pred_default = (y_prob >= 0.5).astype(int)
print(f"  F1 = {f1_score(y_test, y_pred_default, average='binary'):.4f}")

print(f"
Optimal threshold: {best_threshold:.4f}")
print(f"  F1       = {best_f1:.4f}")
print(f"  Precision= {precisions[best_idx]:.4f}")
print(f"  Recall   = {recalls[best_idx]:.4f}")

y_pred = (y_prob >= best_threshold).astype(int)

## Step 7: Evaluation Metrics

In [ ]:
# Cell 8 — Metrics
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["No Rain", "Rain"]))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")

## Step 8: Confusion Matrix

In [ ]:
# Cell 9 — Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: No Rain", "Pred: Rain"],
            yticklabels=["True: No Rain", "True: Rain"])
plt.title(f"Confusion Matrix (threshold={best_threshold:.3f})", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix_stage1.png", dpi=150)
plt.show()
print("Saved → confusion_matrix_stage1.png")

## Step 9: ROC Curve

In [ ]:
# Cell 10 — ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Rain/No-Rain Classifier")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/roc_curve_stage1.png", dpi=150)
plt.show()
print("Saved → roc_curve_stage1.png")

## Step 10: Precision-Recall Curve

In [ ]:
# Cell 11 — Precision-Recall curve
plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, color="darkorange", lw=2)
plt.axvline(x=recalls[best_idx], color="red", linestyle="--",
            label=f"Best threshold = {best_threshold:.3f}
F1 = {best_f1:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Rain/No-Rain Classifier")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pr_curve_stage1.png", dpi=150)
plt.show()
print("Saved → pr_curve_stage1.png")

## Step 11: Feature Importance

In [ ]:
# Cell 12 — Feature importance
feat_imp = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": clf.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 20 Features:")
print(feat_imp.head(20).to_string(index=False))

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp.head(20), x="importance", y="feature", palette="viridis")
plt.title("Top 20 Feature Importances — Stage 1 Classifier")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_importance_stage1.png", dpi=150)
plt.show()
print("Saved → feature_importance_stage1.png")

## Step 12: Save Model & Summary

In [ ]:
# Cell 13 — Save model + summary
clf.save_model(f"{OUTPUT_DIR}/xgb_classifier_stage1.json")

with open(f"{OUTPUT_DIR}/threshold_stage1.txt", "w") as f:
    f.write(str(best_threshold))

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Train samples  : {len(train)}")
print(f"Test samples   : {len(test)}")
print(f"Features used  : {len(FEATURE_COLS)}")
print(f"Best iteration : {clf.best_iteration}")
print(f"ROC-AUC        : {roc_auc:.4f}")
print(f"Best F1        : {best_f1:.4f} @ threshold {best_threshold:.4f}")
print(f"Precision      : {precisions[best_idx]:.4f}")
print(f"Recall         : {recalls[best_idx]:.4f}")
print("=" * 60)
print("Stage 1 complete. Next → Stage 2 regression on rain-only samples.")
print(f"
Saved outputs:")
print(f"  xgb_classifier_stage1.json")
print(f"  scaler_stage1.pkl")
print(f"  threshold_stage1.txt ({best_threshold:.6f})")
print(f"  confusion_matrix_stage1.png")
print(f"  roc_curve_stage1.png")
print(f"  pr_curve_stage1.png")
print(f"  feature_importance_stage1.png")